# 🦜🔗 LangChain ile Büyük Dil Modeli (LLM) Uygulama Geliştirme

Bu çalışmada, modern LLM uygulamalarının endüstri standardı çatısı olan **LangChain** ve **LCEL (LangChain Expression Language)** mimarisi incelenmektedir.

---

## 🧭 LCEL (LangChain Expression Language) Mimarisi
Modern LangChain, Unix boru hattı (`|`) operatörü üzerine kuruludur:

```mermaid
flowchart LR
    A["Girdi Verisi (Input Dict)"] --> B["ChatPromptTemplate"]
    B --> C["ChatModel (Gemini / OpenAI / Groq)"]
    C --> D["OutputParser (Str / JSON / Pydantic)"]
    D --> E["Yapılandırılmış Sonuç"]
```

$$\text{chain} = \text{prompt} \mid \text{llm} \mid \text{StrOutputParser()}$$


In [4]:
import os 
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())



True

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [6]:
llm = ChatGoogleGenerativeAI(model= "gemini-2.0-flash", temperature= 0)

response = llm.invoke("Merhaba nasılsın")
print(response.content)

GoogleModelNotFoundError: Error calling model 'gemini-2.0-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}

In [ ]:
# Prompt Template - parametreli prompt

prompt= ChatPromptTemplate.from_messages([
    ("system", "Sen {rol} rolündesin. {dil} dilinde cevap ver."),
    ("user","{soru}")
])

msg = prompt.incoke({
    "rol": "veri bilimci",
    "dil": "Türkçe",
    "soru": "Overfitting nedir?"
})
print(msg)


In [ ]:
# CHAIN - LCEL ile pipe

chain = prompt | llm | StrOutputParser()

response = chain.invoke({
    "rol": "öğretmen",
    "dil": "Türkçe",
    "soru": "Makina öğrenmesi nedir"
})
print(response)

In [ ]:
from langchain_core.runnables import RunnablePassthrough

secme_prompt = ChatPromptTemplate.from_messages([
    ("system", "En uygun bir yazar "),
    ("user", "{konu} hakkında kısa bir paragraf yaz")
])

yazma_prompt = ChatPromptTemplate.from_messages([
    ("system", "Sen bir {yazar_tipi}isin. Kendi üslubunla yaz."),
    ("user", "{konu} hakkında kısa bir paragraf yaz")
])

secme_chain = secme_prompt | llm | StrOutputParser()

full_chain = (
    {"konu": RunnablePassthrough(), "yazar_tipi": secme_chain}
    | yazma_prompt
    | llm
    | StrOutputParser()  
)

result = full_chain.invoke("uzay araştırmaları")
print(result)